## Embedding Similarity
Given a query embedding and document embeddings, return top-k most similar docs.

In [ ]:
import math
import numpy as np
from typing import List, Any

def cosine_similarity(a, b):
    dot = sum(x * y for x, y in zip(a,b))
    norm_a = math.sqrt(x * x for x in a)
    norm_b = math.sqrt(x * x for x in b)

    if norm_a == 0 or norm_b == 0:
        return 0
    
    return dot/(norm_a * norm_b)

def top_k_similar_docs(query_embedding, documents, k=5, embeddings_fn=None) -> List[Any]:
    scored_docs = []
    for doc in documents:
        doc_embedding = embeddings_fn(doc["text"])
        score = cosine_similarity(query_embedding, doc_embedding)
        scored_docs.append((score, doc))

    sorted(scored_docs, key = lambda x: x[0], reverse = True)

    return [
            {
                "id": docs["id"],
                "text": docs["text"],
                "score": score
            } for score, docs in scored_docs[:k]
        ]
    


## Chunking Strategy

Split long text into overlapping chunks.

In [ ]:
def chunk_text(text, chunk_size=1000, overlap=20):
    
    words = text.split()

    if chunk_size >= len(words) or chunk_size < overlap:
        return [text]
        
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end].strip())

        chunks.append(
            {
                "text": chunk,
                "start": start,
                "end": end,
                "start_word": words[start],
                "end_word": words[end-1]
            }
        )

        start += chunk_size - overlap
    
    return chunks

    return chunks
    

## Retrieval Pipeline
Build a simple RAG retrieval flow.

In [ ]:
from typing import List, Any
import math
import numpy as np

class SimpleRetriever:
    def __init__(self, documents):
        self.documents = documents
        self.vocab = ["this", "is", "a", "test", "document"]
        self.embeddings_fn = self.bow_embedding_fn(self.vocab)

    def bow_embedding_fn(self, vocab):
        word_2_idx = {word : i  for i , word in enumerate(vocab)}

        def embeddings_fn(text):
            vec = np.zeros(len(vocab))
            for word in text.split():
                if word not in word_2_idx:
                    continue
                vec[word_2_idx[word]] += 1
            return vec
        
        return embeddings_fn
    
    def cosine_similarity(self, a, b):
        dot = sum(x * y for x,  y in zip(a,b))
        norm_a = math.sqrt(x * x for x in a)
        norm_b = math.sqrt(x * x for x in b)

        if norm_a == 0 or norm_b == 0:
            return 0
        
        return dot/(norm_a * norm_b)
            
    def top_k_similar_docs(self, query_embedding, k=5):

        scored_docs = []
        for doc in self.documents:
            doc_embedding = self.embeddings_fn(doc["text"])
            score = self.cosine_similarity(query_embedding, doc_embedding)
            scored_docs.append((score, doc))

        sorted(scored_docs, key = lambda x: x[0], reverse = True)

        return [
            {
                "id": docs["id"],
                "text": docs["text"],
                "score": score
            } for score, docs in scored_docs[:k]
        ]

    def retrieve(self, query, k=5):
        query_embedding = self.embeddings_fn(query)
        return top_k_similar_docs(query_embedding, self.documents, k)

class SimpleRagPipeline:
    def __init__(self, documents):
        self.retriever = SimpleRetriever(documents)

    
    def build_context(self, retrieved_docs):
        context = ""
        for doc in retrieved_docs:
            context += f"DocumentId: {doc['id']}\n"
            context += f"Text: {doc['text']}\n"
            context += f"Score: {doc['score']}\n"
            context += f"-----------------------------------\n"

        return context
    
    def rag_pipeline(self, query, k=5):

        retrieved_docs = self.retriever.retrieve(query, k)
        context = self.build_context(retrieved_docs)
        prompt = f"Context: {context}\nQuery: {query}\nAnswer:"
        return prompt


documents = [
    {
        "id": 1,
        "text": "This is a test document."
    }
]
srp = SimpleRagPipeline(documents)



        

## Hydrid Search Ranking
Hybrid search combines lexical search, like BM25, with semantic vector search.


Problem : Given candidate results with:

vector_score
keyword_score
recency_score